# Credit Card Fraud Detection Using Apache Spark and Random Forest

**Course:** Cluster Computing (INT315)  
**Approach:** PySpark + Spark MLlib + one model (Random Forest Classifier)

This notebook is organized into small, sequential cells. Run each cell from top to bottom. In Google Colab, the upload cell will let you select your CSV file directly.

### Project objectives
1. Clean and inspect the transaction dataset.
2. Explore fraud frequency and transaction patterns.
3. Prepare model features while avoiding target leakage.
4. Train **one Random Forest classifier** using Spark MLlib.
5. Evaluate the model with precision, recall, F1-score, ROC-AUC, PR-AUC, and a confusion matrix.
6. Export predictions for review.

> **Important:** `fraud_type` and `fraud_amount_lost` are not allowed as model inputs because they reveal information about the outcome. Raw transaction/customer/card/merchant IDs are also excluded from features.

## Step 1 — Install PySpark

Run this cell once in a fresh Google Colab runtime. It installs a stable PySpark 3.5 release.

In [1]:
%pip -q install pyspark==4.0.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

print("Libraries imported successfully.")

## Step 3 — Upload your CSV file

When you run this cell in Google Colab, click **Choose Files** and select `credit_card_fraud_detection_2026.csv` (or your CSV file name). If you are running the notebook outside Colab, enter the full local path when prompted.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV file was uploaded. Please upload your dataset as a .csv file.")
    csv_path = csv_files[0]
except ImportError:
    csv_path = input("Enter the full path to your CSV file: ").strip()
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"File not found: {csv_path}")

print("Selected file:", csv_path)

## Step 4 — Start Spark and read the CSV

In [ ]:
spark = (
    SparkSession.builder
    .appName("CreditCardFraudDetection")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "")
    .csv(csv_path)
)

print(f"Rows: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")
raw_df.printSchema()

## Step 5 — Preview the data

In [ ]:
display(raw_df.limit(10).toPandas())
print("Column names:")
print(raw_df.columns)

## Step 6 — Check data quality

This cell checks missing values, fully duplicated rows, and whether the target column exists. Missing values will be handled in the preprocessing pipeline rather than deleting many transactions.

In [ ]:
if "is_fraud" not in raw_df.columns:
    raise ValueError("The CSV must contain a column named 'is_fraud'.")

row_count = raw_df.count()
duplicate_count = row_count - raw_df.dropDuplicates().count()

missing_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
]
missing_row = raw_df.select(*missing_exprs).first().asDict()
missing_table = pd.DataFrame(
    [(col_name, int(n_missing), 100 * int(n_missing) / max(row_count, 1))
     for col_name, n_missing in missing_row.items() if n_missing > 0],
    columns=["column", "missing_values", "missing_percent"]
).sort_values("missing_values", ascending=False)

print(f"Total rows: {row_count:,}")
print(f"Fully duplicated rows: {duplicate_count:,}")
if missing_table.empty:
    print("No null values found.")
else:
    display(missing_table)

if "browser" in raw_df.columns:
    print("The browser column is retained; missing categorical values will be handled by StringIndexer(handleInvalid='keep').")

## Step 7 — Inspect the prediction target

`is_fraud = 0` means legitimate and `is_fraud = 1` means fraudulent. Since fraud is a minority class, accuracy alone will not be enough to judge the model.

In [ ]:
target_counts = (
    raw_df.groupBy("is_fraud")
    .agg(F.count("*").alias("transactions"))
    .orderBy("is_fraud")
)
target_pdf = target_counts.toPandas()
total_target_rows = target_pdf["transactions"].sum()
target_pdf["percentage"] = 100 * target_pdf["transactions"] / total_target_rows
display(target_pdf)

plt.figure(figsize=(6, 4))
plt.bar(target_pdf["is_fraud"].astype(str), target_pdf["transactions"])
plt.title("Transaction Class Distribution")
plt.xlabel("is_fraud (0 = legitimate, 1 = fraud)")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

## Step 8 — Explore transaction amounts

In [ ]:
amount_summary = (
    raw_df.groupBy("is_fraud")
    .agg(
        F.count("*").alias("transactions"),
        F.avg("transaction_amount").alias("average_amount"),
        F.expr("percentile_approx(transaction_amount, 0.5)").alias("median_amount")
    )
    .orderBy("is_fraud")
)
amount_pdf = amount_summary.toPandas()
display(amount_pdf)

plt.figure(figsize=(6, 4))
plt.bar(amount_pdf["is_fraud"].astype(str), amount_pdf["average_amount"])
plt.title("Average Transaction Amount by Class")
plt.xlabel("is_fraud (0 = legitimate, 1 = fraud)")
plt.ylabel("Average transaction amount")
plt.tight_layout()
plt.show()

## Step 9 — Explore selected fraud indicators

The observed rates below describe this dataset; they do not prove that a feature causes fraud.

In [ ]:
indicator_columns = [
    c for c in ["high_risk_country", "international_transaction", "card_present", "transaction_channel"]
    if c in raw_df.columns
]

for indicator in indicator_columns:
    print(f"\nFraud rate by {indicator}")
    summary = (
        raw_df.groupBy(indicator)
        .agg(
            F.count("*").alias("transactions"),
            F.sum(F.col("is_fraud").cast("double")).alias("fraud_transactions"),
            F.avg(F.col("is_fraud").cast("double")).alias("fraud_rate")
        )
        .withColumn("fraud_rate_percent", F.round(F.col("fraud_rate") * 100, 3))
        .orderBy(F.desc("fraud_rate"))
    )
    display(summary)

## Step 10 — Prepare data and avoid target leakage

These columns are excluded **from the model features**:
- Unique identifiers: `transaction_id`, `customer_id`, `card_id`, `merchant_id`
- Raw timestamp string: `transaction_datetime` (the dataset already has hour/day/weekend fields)
- Outcome-revealing fields: `fraud_type`, `fraud_amount_lost`

They remain in the original DataFrame where useful for reviewing predictions, but are not supplied to the classifier.

In [ ]:
# Keep the original columns available for interpretation and export.
df = raw_df.withColumn("is_fraud", F.col("is_fraud").cast("double"))

excluded_from_features = [
    "transaction_id",
    "customer_id",
    "card_id",
    "merchant_id",
    "transaction_datetime",
    "fraud_type",
    "fraud_amount_lost",
    "is_fraud"
]

available_exclusions = [c for c in excluded_from_features if c in df.columns]
feature_fields = [field for field in df.schema.fields if field.name not in available_exclusions]
numeric_cols = [field.name for field in feature_fields if isinstance(field.dataType, NumericType)]
categorical_cols = [field.name for field in feature_fields if isinstance(field.dataType, StringType)]

print(f"Numeric feature columns ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nCategorical feature columns ({len(categorical_cols)}):")
print(categorical_cols)

if not numeric_cols and not categorical_cols:
    raise ValueError("No usable features were found after excluding target/leakage/ID columns.")

valid_labels = df.filter(F.col("is_fraud").isin(0.0, 1.0))
invalid_label_count = df.count() - valid_labels.count()
if invalid_label_count:
    print(f"Removing {invalid_label_count} rows with missing or invalid is_fraud labels.")
df = valid_labels

## Step 11 — Split training and test data

The test set is held out until evaluation. The training set is used to calculate class weights so that the minority fraud class receives more importance during training.

In [ ]:
train_df, test_df = df.randomSplit([0.80, 0.20], seed=42)
train_count = train_df.count()
test_count = test_df.count()

fraud_count = train_df.filter(F.col("is_fraud") == 1.0).count()
legitimate_count = train_df.filter(F.col("is_fraud") == 0.0).count()

if fraud_count == 0 or legitimate_count == 0:
    raise ValueError("The training split must contain both fraud and legitimate transactions.")

# Inverse-frequency weights, calculated on training data only.
legitimate_weight = train_count / (2.0 * legitimate_count)
fraud_weight = train_count / (2.0 * fraud_count)

train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1.0, F.lit(fraud_weight)).otherwise(F.lit(legitimate_weight))
)
test_weighted = test_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1.0, F.lit(fraud_weight)).otherwise(F.lit(legitimate_weight))
)

print(f"Training rows: {train_count:,}")
print(f"Testing rows: {test_count:,}")
print(f"Fraud rows in training: {fraud_count:,}")
print(f"Legitimate rows in training: {legitimate_count:,}")
print(f"Legitimate class weight: {legitimate_weight:.4f}")
print(f"Fraud class weight: {fraud_weight:.4f}")

## Step 12 — Build the preprocessing pipeline

Numerical missing values are imputed with the median. Categorical columns are indexed and one-hot encoded. `handleInvalid="keep"` allows missing or previously unseen categories to be processed without dropping the transaction.

In [ ]:
imputed_numeric_cols = [f"imputed__{c}" for c in numeric_cols]
indexer_output_cols = [f"indexed__{c}" for c in categorical_cols]
encoded_categorical_cols = [f"encoded__{c}" for c in categorical_cols]

stages = []

if numeric_cols:
    imputer = Imputer(
        strategy="median",
        inputCols=numeric_cols,
        outputCols=imputed_numeric_cols
    )
    stages.append(imputer)

indexers = [
    StringIndexer(inputCol=source, outputCol=target, handleInvalid="keep")
    for source, target in zip(categorical_cols, indexer_output_cols)
]
stages.extend(indexers)

if categorical_cols:
    encoder = OneHotEncoder(
        inputCols=indexer_output_cols,
        outputCols=encoded_categorical_cols,
        handleInvalid="keep"
    )
    stages.append(encoder)

assembler_inputs = imputed_numeric_cols + encoded_categorical_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)
stages.append(assembler)

print("Preprocessing stages ready.")
print("Number of assembled feature groups:", len(assembler_inputs))

## Step 13 — Define the one prediction model

We will use **only Random Forest Classifier**. It supports nonlinear patterns in tabular data. The `class_weight` column compensates for the dataset's imbalance during model fitting.

In [ ]:
random_forest = RandomForestClassifier(
    labelCol="is_fraud",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=10,
    seed=42
)
stages.append(random_forest)

pipeline = Pipeline(stages=stages)
print("Pipeline configured with one prediction model: Random Forest Classifier.")

## Step 14 — Train the model

This can take a few minutes depending on the runtime. The preprocessing stages are fitted on the training data only.

In [ ]:
pipeline_model = pipeline.fit(train_weighted)
print("Training complete.")

## Step 15 — Generate test predictions

In [ ]:
predictions = pipeline_model.transform(test_weighted)

prediction_view = (
    predictions
    .withColumn("fraud_probability", vector_to_array(F.col("probability"))[1])
    .select("transaction_id", "is_fraud", "prediction", "fraud_probability")
)

display(prediction_view.limit(20).toPandas())

## Step 16 — Evaluate model performance

- **Precision:** Of transactions flagged as fraud, how many were actually fraud?
- **Recall:** Of all actual fraud transactions, how many did the model detect?
- **F1-score:** Harmonic mean of precision and recall.
- **PR-AUC:** Particularly useful when the positive class is rare.
- **ROC-AUC:** Measures ranking performance across classification thresholds.

In [ ]:
tp = predictions.filter((F.col("is_fraud") == 1.0) & (F.col("prediction") == 1.0)).count()
fp = predictions.filter((F.col("is_fraud") == 0.0) & (F.col("prediction") == 1.0)).count()
tn = predictions.filter((F.col("is_fraud") == 0.0) & (F.col("prediction") == 0.0)).count()
fn = predictions.filter((F.col("is_fraud") == 1.0) & (F.col("prediction") == 0.0)).count()

total_predictions = tp + fp + tn + fn
accuracy = (tp + tn) / total_predictions if total_predictions else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)
roc_auc = roc_evaluator.evaluate(predictions)
pr_auc = pr_evaluator.evaluate(predictions)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc, pr_auc]
})
display(metrics_df.style.format({"Score": "{:.4f}"}))

print("Confusion matrix counts:")
print(f"True Positives (fraud correctly detected): {tp:,}")
print(f"False Positives (legitimate flagged as fraud): {fp:,}")
print(f"True Negatives (legitimate correctly identified): {tn:,}")
print(f"False Negatives (fraud missed): {fn:,}")

## Step 17 — Visualize the confusion matrix

In [ ]:
confusion = np.array([[tn, fp], [fn, tp]])

plt.figure(figsize=(5.5, 4.5))
plt.imshow(confusion, interpolation="nearest")
plt.title("Random Forest — Confusion Matrix")
plt.colorbar(label="Number of transactions")
plt.xticks([0, 1], ["Predicted Legitimate", "Predicted Fraud"], rotation=15)
plt.yticks([0, 1], ["Actual Legitimate", "Actual Fraud"])
for row in range(2):
    for col in range(2):
        plt.text(col, row, f"{confusion[row, col]:,}", ha="center", va="center")
plt.tight_layout()
plt.show()

## Step 18 — Review the most important model features

Feature importance is reported at the transformed feature level. One-hot encoded categories may appear as individual category indicators.

In [ ]:
rf_model = pipeline_model.stages[-1]
feature_metadata = predictions.schema["features"].metadata.get("ml_attr", {}).get("attrs", {})
feature_names_by_index = {}

for _, attributes in feature_metadata.items():
    for attribute in attributes:
        if "idx" in attribute and "name" in attribute:
            feature_names_by_index[int(attribute["idx"])] = attribute["name"]

importance_values = rf_model.featureImportances.toArray()
importance_rows = [
    (feature_names_by_index.get(i, f"feature_{i}"), float(score))
    for i, score in enumerate(importance_values)
]
importance_pdf = pd.DataFrame(importance_rows, columns=["feature", "importance"])
importance_pdf = importance_pdf.sort_values("importance", ascending=False).head(15)
display(importance_pdf)

plt.figure(figsize=(9, 6))
plt.barh(importance_pdf["feature"][::-1], importance_pdf["importance"][::-1])
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Step 19 — Export predictions to CSV

This exports predictions for the held-out test set, including the transaction ID, actual label, predicted label, and estimated fraud probability. In Colab, the file will appear in the Files panel.

In [ ]:
output_dir = "/content" if os.path.isdir("/content") else os.getcwd()
output_path = os.path.join(output_dir, "fraud_predictions.csv")

export_pdf = prediction_view.toPandas()
export_pdf.to_csv(output_path, index=False)

print(f"Saved {len(export_pdf):,} predictions to: {output_path}")
print("Open the Colab Files panel on the left to download fraud_predictions.csv.")

## Step 20 — Project conclusion template

After running the notebook, complete the summary using your actual results:

- The dataset contains **[row count]** transactions and **[fraud count]** labelled fraudulent transactions.
- The model used was **Random Forest Classifier from Spark MLlib**.
- Test precision: **[precision]**; test recall: **[recall]**; test F1-score: **[F1]**; test PR-AUC: **[PR-AUC]**.
- The confusion matrix shows how many fraud cases were detected and missed, along with how many legitimate transactions were incorrectly flagged.
- This is an educational prototype, not a production payment-blocking system. A real deployment would need validation on time-separated data, operational monitoring, threshold selection, privacy and fairness checks, and review by payment-risk experts.

## Optional — Stop Spark when finished

Run this only after all analysis and exports are complete.

In [ ]:
# spark.stop()